# Use Larger LLMs for Task Planning, Smaller LLMs for Execution

Not all tasks require the same level of intelligence. A password reset question doesn't need the same reasoning power as a complex billing dispute. In this notebook, we build a **customer support triage system** that uses:

- A **large LLM** (Claude Sonnet) to classify tickets and decide the resolution path
- A **small LLM** (Mistral Small) to handle simple, templated responses quickly and cheaply
- The **large LLM** again for complex reasoning and final quality review

Elastic Managed LLMs are [billed per million tokens](https://www.elastic.co/docs/reference/kibana/connectors-kibana/elastic-managed-llm), so routing simple tickets to a smaller, cheaper model has a real impact on costs.

We orchestrate this with **Elasticsearch Workflows**, using Kibana AI Connectors to manage both model endpoints.

## Setup

In [1]:
%pip install -qU elasticsearch python-dotenv requests


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from dotenv import load_dotenv
import os
import requests
import json

load_dotenv()

ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
KIBANA_URL = os.getenv("KIBANA_URL")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

## Connect to Elasticsearch

In [3]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY)
es_client.info()

ObjectApiResponse({'name': 'instance-0000000005', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Set up AI Connectors

We use two AI Connectors for the workflow:

| Connector | Model | Type | Role | Why |
| :--- | :--- | :--- | :--- | :--- |
| `Anthropic Claude Sonnet 4.6` | Claude Sonnet | Elastic Managed | Classification, complex reasoning, review | Strong reasoning for ambiguous tickets and nuanced issues |
| `Mistral Small` | mistral-small-latest | Custom (OpenAI-compatible) | Simple, templated responses | Fast and cheap for straightforward responses |

The Sonnet connector is already available as an Elastic Managed LLM. We only need to create a custom connector for Mistral.

In [5]:
SMALL_LLM_CONNECTOR = "Mistral Small"
headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

# Create custom Mistral connector (OpenAI-compatible)
mistral_connector_payload = {
    "connector_type_id": ".gen-ai",
    "name": SMALL_LLM_CONNECTOR,
    "config": {
        "apiProvider": "Other",
        "apiUrl": "https://api.mistral.ai/v1/chat/completions",
        "defaultModel": "mistral-small-latest",
    },
    "secrets": {
        "apiKey": MISTRAL_API_KEY,
    },
}

# Let Kibana generate the UUID automatically
response = requests.post(
    f"{KIBANA_URL}/api/actions/connector",
    headers=headers,
    json=mistral_connector_payload,
)
result = response.json()
MISTRAL_CONNECTOR_ID = result.get("id")
print(f"Mistral connector created: {response.status_code}")
print(f"Connector ID: {MISTRAL_CONNECTOR_ID}")
print(result)

Mistral connector created: 200
Connector ID: 71a1d1b6-0fc9-4e4d-89eb-7b854b2f7b53
{'id': '71a1d1b6-0fc9-4e4d-89eb-7b854b2f7b53', 'name': 'Mistral Small', 'config': {'apiProvider': 'Other', 'apiUrl': 'https://api.mistral.ai/v1/chat/completions', 'defaultModel': 'mistral-small-latest'}, 'connector_type_id': '.gen-ai', 'is_preconfigured': False, 'is_deprecated': False, 'is_missing_secrets': False, 'is_system_action': False, 'is_connector_type_deprecated': False}


## Index the knowledge base

We index a set of support articles so the workflow can search for relevant context before generating responses. This grounds the LLM answers in real documentation instead of relying on the model's general knowledge.

In [ ]:
INDEX_NAME = "support-knowledge-base"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "title": {"type": "text"},
            "category": {"type": "keyword"},
            "content": {"type": "text"},
            "tags": {"type": "keyword"},
        }
    },
)
print(f"Created index: {INDEX_NAME}")

Created index: support-knowledge-base


In [22]:
from elasticsearch import helpers

with open("dataset.json") as f:
    articles = json.load(f)


def build_bulk_actions(articles, index_name):
    for i, article in enumerate(articles):
        yield {
            "_index": index_name,
            "_id": i,
            "_source": article,
        }


success, failed = helpers.bulk(
    es_client,
    build_bulk_actions(articles, INDEX_NAME),
    refresh=True,
)
print(f"Indexed {success} articles into '{INDEX_NAME}'")

Indexed 15 articles into 'support-knowledge-base'


## The Workflow

Our workflow follows this pattern:

```
                    ┌─────────────────────┐
                    │   Customer Ticket    │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Classify Ticket    │
                    │  (Claude Sonnet)     │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │  Search Knowledge   │
                    │  Base (Elasticsearch)│
                    └──────────┬──────────┘
                               │
                 ┌─────────────┴─────────────┐
                 │                           │
          complexity:                 complexity:
           simple                      complex
                 │                           │
      ┌──────────▼──────────┐   ┌───────────▼──────────┐
      │  Generate Response  │   │   Deep Analysis &    │
      │  (Mistral Small)    │   │     Response         │
      │   Fast & cheap      │   │  (Claude Sonnet)     │
      └──────────┬──────────┘   └───────────┬──────────┘
                 │                           │
                 └─────────────┬─────────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Review & Polish    │
                    │  (Claude Sonnet)     │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Final Response     │
                    └─────────────────────┘
```

### Elasticsearch Workflow definition

The following YAML defines the workflow in Elasticsearch. Note that Elasticsearch Workflows don't currently have a REST API, so this is configured directly in the Workflow UI.

```yaml
name: support_ticket_triage
description: >
  Routes customer support tickets through LLM-powered triage.
  Uses Claude Sonnet for classification and complex reasoning,
  Mistral Small for simple responses, and Claude Sonnet again
  for quality review. Searches a knowledge base for context.
enabled: true

inputs:
  - name: ticket_text
    type: string
    description: The customer support ticket text
    required: true

consts:
  indexName: support-knowledge-base

triggers:
  - type: manual

steps:
  # Step 1: Classify the ticket using Claude Sonnet
  - name: classify_ticket
    type: ai.prompt
    with:
      connectorId: Anthropic Claude Sonnet 4.6
      prompt: >
        You are a customer support ticket classifier.
        Analyze the following ticket and return ONLY a JSON object with:
        - "category": one of ["password_reset", "order_status", "billing_issue", "technical_issue", "general_inquiry"]
        - "complexity": one of ["simple", "complex"]
        - "summary": brief one-line summary of the issue

        Classify as "complex" if the ticket involves multiple issues,
        requires investigation, or has an emotional/urgent tone.

        Ticket: {{ inputs.ticket_text }}

  # Step 2: Search the knowledge base for relevant articles
  - name: search_knowledge_base
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        bool:
          should:
            - match:
                content: "{{ inputs.ticket_text }}"
            - match:
                title: "{{ inputs.ticket_text }}"
      size: 3

  # Step 3: Route based on complexity
  - name: route_by_complexity
    type: if
    condition: "${{ steps.classify_ticket.output.complexity == 'simple' }}"
    steps:
      # Simple path: use Mistral Small (fast & cheap)
      - name: generate_simple_response
        type: ai.prompt
        with:
          connectorId: Mistral Small
          prompt: >
            You are a customer support agent. Generate a helpful, concise response
            for this ticket using the knowledge base articles below as context.
            Be friendly, professional, and include specific next steps.

            Ticket: {{ inputs.ticket_text }}

            Knowledge base articles:
            {{ steps.search_knowledge_base.output.hits.hits | json }}
    else:
      # Complex path: use Claude Sonnet (deep reasoning)
      - name: generate_complex_response
        type: ai.prompt
        with:
          connectorId: Anthropic Claude Sonnet 4.6
          prompt: >
            You are a senior customer support specialist. This ticket requires careful analysis.
            Use the knowledge base articles below as context to provide an accurate response.
            Provide a detailed, empathetic response that:
            1. Acknowledges all concerns raised
            2. Explains what happened (if applicable)
            3. Provides clear resolution steps based on our documentation
            4. Offers follow-up options

            Ticket: {{ inputs.ticket_text }}

            Knowledge base articles:
            {{ steps.search_knowledge_base.output.hits.hits | json }}

  # Step 4: Review the response using Claude Sonnet
  - name: review_response
    type: ai.prompt
    with:
      connectorId: Anthropic Claude Sonnet 4.6
      prompt: >
        Review this customer support response for quality.
        Ensure it is professional, accurate, empathetic, and
        addresses all concerns from the original ticket.
        Return only the final, polished response.

        Original ticket: {{ inputs.ticket_text }}
        Draft response: {{ steps.route_by_complexity.output }}
```

## Using the Workflow as a tool in Agent Builder

The classification works as expected — the large LLM correctly identifies simple vs. complex tickets. Now, instead of reimplementing the workflow logic in Python, we use the **Elastic Agent Builder** to wire everything together.

The Agent Builder allows you to create AI agents with access to **tools**, and an Elasticsearch Workflow can be registered as one of those tools. The agent receives the customer ticket, calls the workflow tool (which handles the full triage pipeline), and presents the result.

We use the [Agent Builder Kibana APIs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api) to create the tools and the agent.

**Note:** After creating the workflow in the Kibana UI using the YAML above, copy its ID and set it below as `WORKFLOW_ID`.

In [6]:
WORKFLOW_ID = "workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae"  # Copy from Elastic UI after creating the workflow

In [7]:
# Create the workflow tool
workflow_tool_payload = {
    "id": "run_support_triage_workflow",
    "type": "workflow",
    "description": (
        "Triggers the support ticket triage workflow. "
        "Use this tool to process a customer support ticket. "
        "The workflow classifies the ticket, routes it to the appropriate model "
        "(small LLM for simple tickets, large LLM for complex ones), "
        "generates a response, and reviews it for quality."
    ),
    "tags": ["support", "triage", "workflow"],
    "configuration": {
        "workflow_id": WORKFLOW_ID,
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=workflow_tool_payload,
)
print(f"Workflow tool created: {response.status_code}")
print(response.json())

Workflow tool created: 200
{'id': 'run_support_triage_workflow', 'type': 'workflow', 'description': 'Triggers the support ticket triage workflow. Use this tool to process a customer support ticket. The workflow classifies the ticket, routes it to the appropriate model (small LLM for simple tickets, large LLM for complex ones), generates a response, and reviews it for quality.', 'tags': ['support', 'triage', 'workflow'], 'configuration': {'workflow_id': 'workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae'}, 'readonly': False, 'schema': {'type': 'object', 'properties': {'ticket_text': {'type': 'string', 'description': 'The customer support ticket text'}}, 'required': ['ticket_text'], 'additionalProperties': False, 'description': 'Parameters needed to execute the workflow', '$schema': 'http://json-schema.org/draft-07/schema#'}}


In [8]:
# Create the agent with the workflow tool
agent_payload = {
    "id": "support-triage-agent",
    "name": "Support Triage Agent",
    "description": "Customer support agent that triages tickets using a multi-model workflow.",
    "labels": ["support", "triage"],
    "configuration": {
        "instructions": (
            "You are a customer support assistant. When a customer sends a support ticket, "
            "use the `run_support_triage_workflow` tool to process it.\n\n"
            "The workflow will:\n"
            "1. Classify the ticket (category and complexity) using a large LLM\n"
            "2. Route it to the appropriate model (small LLM for simple, large LLM for complex)\n"
            "3. Generate and review a response\n\n"
            "Present the final response to the customer. If the classification indicates "
            "a 'complex' ticket, also mention that a senior agent has been notified for follow-up."
        ),
        "tools": [{"tool_ids": ["run_support_triage_workflow"]}],
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(response.json())

Agent created: 200
{'id': 'support-triage-agent', 'type': 'chat', 'name': 'Support Triage Agent', 'description': 'Customer support agent that triages tickets using a multi-model workflow.', 'labels': ['support', 'triage'], 'configuration': {'instructions': "You are a customer support assistant. When a customer sends a support ticket, use the `run_support_triage_workflow` tool to process it.\n\nThe workflow will:\n1. Classify the ticket (category and complexity) using a large LLM\n2. Route it to the appropriate model (small LLM for simple, large LLM for complex)\n3. Generate and review a response\n\nPresent the final response to the customer. If the classification indicates a 'complex' ticket, also mention that a senior agent has been notified for follow-up.", 'tools': [{'tool_ids': ['run_support_triage_workflow']}]}, 'readonly': False}


### Test the agent

Let's send support tickets to the agent and see how the workflow routes them.

In [11]:
# Test with a simple ticket
simple_converse = {
    "agent_id": "support-triage-agent",
    "input": "Hi, I forgot my password and can't log in. How do I reset it?",
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=simple_converse,
)
print("Simple ticket response:")
print(json.dumps(response.json(), indent=2))

Simple ticket response:
{
  "conversation_id": "8b530272-0742-43e5-a42b-78fba67f1d37",
  "steps": [
    {
      "type": "reasoning",
      "reasoning": "The user is asking for help with a forgotten password, which is a support ticket. I should use the `run_support_triage_workflow` tool to process it."
    },
    {
      "type": "tool_call",
      "tool_call_id": "tool_run_support_triage_workflow_Krogw3hgkJ3oskTwazyQ",
      "tool_id": "run_support_triage_workflow",
      "progression": [],
      "params": {
        "ticket_text": "Hi, I forgot my password and can't log in. How do I reset it?"
      },
      "results": [
        {
          "type": "other",
          "data": {
            "execution": {
              "execution_id": "824d681a-82aa-4d88-85fd-a1762cfd45b8",
              "status": "completed",
              "workflow_id": "workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae",
              "started_at": "2026-04-09T20:27:49.853Z",
              "finished_at": "2026-04-09T20:28:

In [12]:
# Test with a complex ticket
complex_converse = {
    "agent_id": "support-triage-agent",
    "input": (
        "I've been charged twice for my annual subscription - $299 on March 1st "
        "and again $299 on March 3rd. My bank shows both as completed but your site "
        "only shows one payment. Also, my account was downgraded from Premium to Basic "
        "yesterday even though I clearly paid. I've been a customer for 5 years and "
        "this is really frustrating. I need this resolved ASAP as I use the premium "
        "features for my business."
    ),
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=complex_converse,
)
print("Complex ticket response:")
print(json.dumps(response.json(), indent=2))

Complex ticket response:
{
  "conversation_id": "4f12c005-a964-4c4c-a1d7-32190dc0ad4a",
  "steps": [
    {
      "type": "reasoning",
      "reasoning": "The user is reporting a billing issue and account downgrade, which requires processing as a support ticket."
    },
    {
      "type": "tool_call",
      "tool_call_id": "tool_run_support_triage_workflow_AFmwFFfuUQP7QMMSuCAE",
      "tool_id": "run_support_triage_workflow",
      "progression": [],
      "params": {
        "ticket_text": "I've been charged twice for my annual subscription - $299 on March 1st and again $299 on March 3rd. My bank shows both as completed but your site only shows one payment. Also, my account was downgraded from Premium to Basic yesterday even though I clearly paid. I've been a customer for 5 years and this is really frustrating. I need this resolved ASAP as I use the premium features for my business."
      },
      "results": [
        {
          "type": "other",
          "data": {
            "exec

## Cleanup

In [ ]:
# Delete Agent Builder resources
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/support-triage-agent", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/run_support_triage_workflow", headers=headers
)
print("Deleted Agent Builder resources")

# Delete custom Mistral connector
requests.delete(
    f"{KIBANA_URL}/api/actions/connector/{MISTRAL_CONNECTOR_ID}", headers=headers
)
print(f"Deleted connector: {MISTRAL_CONNECTOR_ID}")

# Delete knowledge base index
es_client.indices.delete(index=INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")